In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from itertools import combinations
from statsmodels.tsa.stattools import adfuller

In [ ]:
minimumAdv = 10

In [32]:
tpxData = pd.read_csv('TPX_prices.csv', index_col=0, parse_dates=True)
# tpxData

In [33]:
tpxData = tpxData.dropna(axis='columns')
# tpxData

In [34]:
tpxUniverseData = pd.read_excel('TPX Universe.xlsx')
# tpxUniverseData

In [35]:
tickersName = tpxData.columns

In [36]:
for tickers in tickersName:
    if (tpxUniverseData.loc[tpxUniverseData['Ticker'] == tickers]['Avg of Daily Equity Traded Val over 3 Months'] < minimumAdv).bool():
        tpxData = tpxData.drop([tickers], axis=1)

/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2679211644.py:2: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (tpxUniverseData.loc[tpxUniverseData['Ticker'] == tickers]['Avg of Daily Equity Traded Val over 3 Months'] < 10).bool():
/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2679211644.py:2: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (tpxUniverseData.loc[tpxUniverseData['Ticker'] == tickers]['Avg of Daily Equity Traded Val over 3 Months'] < 10).bool():
/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2679211644.py:2: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (tpxUniverseData.loc[tpxUniverseData['Ticker'] == tickers]['Avg of Daily Equity Traded Val over 3 Months'] < 10).bool():
/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2679211644.py:2: FutureWarning: 

In [37]:
instrumentList = list(tpxData.columns)
pair_list = list(sorted(map(sorted, combinations(set(instrumentList), 2))))
len(pair_list)

63190

In [38]:
pair_list

[['1332 JP Equity', '1514 JP Equity'],
 ['1332 JP Equity', '1605 JP Equity'],
 ['1332 JP Equity', '1662 JP Equity'],
 ['1332 JP Equity', '1721 JP Equity'],
 ['1332 JP Equity', '1801 JP Equity'],
 ['1332 JP Equity', '1802 JP Equity'],
 ['1332 JP Equity', '1803 JP Equity'],
 ['1332 JP Equity', '1808 JP Equity'],
 ['1332 JP Equity', '1812 JP Equity'],
 ['1332 JP Equity', '1878 JP Equity'],
 ['1332 JP Equity', '1911 JP Equity'],
 ['1332 JP Equity', '1925 JP Equity'],
 ['1332 JP Equity', '1928 JP Equity'],
 ['1332 JP Equity', '1944 JP Equity'],
 ['1332 JP Equity', '1959 JP Equity'],
 ['1332 JP Equity', '1963 JP Equity'],
 ['1332 JP Equity', '1969 JP Equity'],
 ['1332 JP Equity', '2002 JP Equity'],
 ['1332 JP Equity', '2127 JP Equity'],
 ['1332 JP Equity', '2181 JP Equity'],
 ['1332 JP Equity', '2212 JP Equity'],
 ['1332 JP Equity', '2222 JP Equity'],
 ['1332 JP Equity', '2267 JP Equity'],
 ['1332 JP Equity', '2269 JP Equity'],
 ['1332 JP Equity', '2282 JP Equity'],
 ['1332 JP Equity', '2371

In [39]:
# (tpxData.pct_change()+1).cumprod().plot(figsize=(25, 20), legend=False)

# plt.ylabel("Percentage Change")
# plt.show()

In [40]:
def findHedgeRatio(x, y):
    """
    Calculates the hedge ratio between two variables.

    Parameters:
    x: panda dataframe timeseries for instrument A
    y: panda dataframe timeseries for instrument B

    Returns:
    float: The hedge ratio.

    """
    model = sm.OLS(x, sm.add_constant(y)).fit()
    return model.params

In [41]:
# def findSpread(x, y):
#     """
#     Calculates the spread between two variables.

#     Parameters:
#     x: panda dataframe timeseries for instrument A
#     y: panda dataframe timeseries for instrument B

#     Returns:
#     panda dataframe: The spread between the two instruments.

#     """
#     hedgeRatio = findHedgeRatio(x, y)
#     spread = x - hedgeRatio * y
#     return spread

In [42]:
def ADFisStionaryTest(spread):
    """
    Augmented Dickey-Fuller test for stationarity.

    Parameters:
    spread: panda dataframe timeseries

    Returns:
    True if pair is stationary
    False if pair is not stationary
    """
    result = adfuller(spread, maxlag=1)
    # print('ADF Statistic: %f' % result[0])
    # print('p-value: %f' % result[1])
    # print('Critical Values:')
    # for key, value in result[4].items():
    #     print('\t%s: %.3f' % (key, value))
    if (result[0] < result[4]['1%']):
        return True
    else:
        return False

In [43]:
validPairsList = []

### Find valid pairs and its spread

In [44]:
# ### DO NOT REMOVE

validPairs = pd.DataFrame()
for pair in pair_list:
    dfSpread = pd.DataFrame()
    hedgeRatioParams = findHedgeRatio(tpxData[pair[0]], tpxData[pair[1]])
    dfSpread['spread'] = tpxData[pair[0]] - hedgeRatioParams[1] * tpxData[pair[1]] - hedgeRatioParams[0]
    adfResult = ADFisStionaryTest(dfSpread.spread)
    if adfResult == True:
        validPairs[f'spread_{pair[0]}_{pair[1]}'] = dfSpread['spread']
        validPairsList.append(pair)
    else:
        print(f"{pair[0]} and {pair[1]} pair is not stationary")

9508 JP Equity and 9602 JP Equity pair is not stationary
9508 JP Equity and 9603 JP Equity pair is not stationary
9508 JP Equity and 9613 JP Equity pair is not stationary
9508 JP Equity and 9616 JP Equity pair is not stationary
9508 JP Equity and 9684 JP Equity pair is not stationary
9508 JP Equity and 9697 JP Equity pair is not stationary
9508 JP Equity and 9706 JP Equity pair is not stationary
9508 JP Equity and 9719 JP Equity pair is not stationary
9508 JP Equity and 9735 JP Equity pair is not stationary
9508 JP Equity and 9766 JP Equity pair is not stationary
9508 JP Equity and 9831 JP Equity pair is not stationary
9508 JP Equity and 9843 JP Equity pair is not stationary
9508 JP Equity and 9861 JP Equity pair is not stationary
9508 JP Equity and 9962 JP Equity pair is not stationary
9508 JP Equity and 9983 JP Equity pair is not stationary
9508 JP Equity and 9984 JP Equity pair is not stationary
9509 JP Equity and 9513 JP Equity pair is not stationary
9509 JP Equity and 9531 JP Equi

/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfSpread['spread'] = tpxData[pair[0]] - hedgeRatioParams[1] * tpxData[pair[1]] - hedgeRatioParams[0]
/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfSpread['spread'] = tpxData[pair[0]] - hedgeRatioParams[1] * tpxData[pair[1]] - hedgeRatioParams[0]
/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a fut

9509 JP Equity and 9684 JP Equity pair is not stationary
9509 JP Equity and 9697 JP Equity pair is not stationary
9509 JP Equity and 9706 JP Equity pair is not stationary
9509 JP Equity and 9719 JP Equity pair is not stationary
9509 JP Equity and 9735 JP Equity pair is not stationary
9509 JP Equity and 9766 JP Equity pair is not stationary
9509 JP Equity and 9831 JP Equity pair is not stationary
9509 JP Equity and 9843 JP Equity pair is not stationary
9509 JP Equity and 9861 JP Equity pair is not stationary
9509 JP Equity and 9962 JP Equity pair is not stationary
9509 JP Equity and 9983 JP Equity pair is not stationary
9509 JP Equity and 9984 JP Equity pair is not stationary
9513 JP Equity and 9531 JP Equity pair is not stationary
9513 JP Equity and 9532 JP Equity pair is not stationary
9513 JP Equity and 9602 JP Equity pair is not stationary
9513 JP Equity and 9603 JP Equity pair is not stationary
9513 JP Equity and 9613 JP Equity pair is not stationary
9513 JP Equity and 9616 JP Equi

/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfSpread['spread'] = tpxData[pair[0]] - hedgeRatioParams[1] * tpxData[pair[1]] - hedgeRatioParams[0]
/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfSpread['spread'] = tpxData[pair[0]] - hedgeRatioParams[1] * tpxData[pair[1]] - hedgeRatioParams[0]
/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a fut

9513 JP Equity and 9766 JP Equity pair is not stationary
9513 JP Equity and 9831 JP Equity pair is not stationary
9513 JP Equity and 9843 JP Equity pair is not stationary
9513 JP Equity and 9861 JP Equity pair is not stationary
9513 JP Equity and 9983 JP Equity pair is not stationary
9513 JP Equity and 9984 JP Equity pair is not stationary
9531 JP Equity and 9532 JP Equity pair is not stationary
9531 JP Equity and 9602 JP Equity pair is not stationary
9531 JP Equity and 9603 JP Equity pair is not stationary
9531 JP Equity and 9613 JP Equity pair is not stationary
9531 JP Equity and 9616 JP Equity pair is not stationary
9531 JP Equity and 9684 JP Equity pair is not stationary
9531 JP Equity and 9697 JP Equity pair is not stationary
9531 JP Equity and 9706 JP Equity pair is not stationary
9531 JP Equity and 9719 JP Equity pair is not stationary
9531 JP Equity and 9735 JP Equity pair is not stationary
9531 JP Equity and 9766 JP Equity pair is not stationary
9531 JP Equity and 9831 JP Equi

...

/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfSpread['spread'] = tpxData[pair[0]] - hedgeRatioParams[1] * tpxData[pair[1]] - hedgeRatioParams[0]
/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfSpread['spread'] = tpxData[pair[0]] - hedgeRatioParams[1] * tpxData[pair[1]] - hedgeRatioParams[0]
/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a fut

9684 JP Equity and 9861 JP Equity pair is not stationary
9684 JP Equity and 9983 JP Equity pair is not stationary
9684 JP Equity and 9984 JP Equity pair is not stationary
9697 JP Equity and 9706 JP Equity pair is not stationary
9697 JP Equity and 9719 JP Equity pair is not stationary
9697 JP Equity and 9735 JP Equity pair is not stationary
9697 JP Equity and 9766 JP Equity pair is not stationary
9697 JP Equity and 9831 JP Equity pair is not stationary
9697 JP Equity and 9843 JP Equity pair is not stationary
9697 JP Equity and 9861 JP Equity pair is not stationary
9697 JP Equity and 9962 JP Equity pair is not stationary
9697 JP Equity and 9984 JP Equity pair is not stationary
9706 JP Equity and 9719 JP Equity pair is not stationary
9706 JP Equity and 9735 JP Equity pair is not stationary
9706 JP Equity and 9766 JP Equity pair is not stationary
9706 JP Equity and 9831 JP Equity pair is not stationary
9706 JP Equity and 9843 JP Equity pair is not stationary
9706 JP Equity and 9861 JP Equi

/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfSpread['spread'] = tpxData[pair[0]] - hedgeRatioParams[1] * tpxData[pair[1]] - hedgeRatioParams[0]
/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfSpread['spread'] = tpxData[pair[0]] - hedgeRatioParams[1] * tpxData[pair[1]] - hedgeRatioParams[0]
/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a fut

9719 JP Equity and 9831 JP Equity pair is not stationary
9719 JP Equity and 9843 JP Equity pair is not stationary
9719 JP Equity and 9962 JP Equity pair is not stationary
9719 JP Equity and 9983 JP Equity pair is not stationary
9719 JP Equity and 9984 JP Equity pair is not stationary
9735 JP Equity and 9766 JP Equity pair is not stationary
9735 JP Equity and 9831 JP Equity pair is not stationary
9735 JP Equity and 9962 JP Equity pair is not stationary
9735 JP Equity and 9983 JP Equity pair is not stationary
9735 JP Equity and 9984 JP Equity pair is not stationary
9766 JP Equity and 9831 JP Equity pair is not stationary
9766 JP Equity and 9843 JP Equity pair is not stationary
9766 JP Equity and 9861 JP Equity pair is not stationary
9766 JP Equity and 9962 JP Equity pair is not stationary
9766 JP Equity and 9983 JP Equity pair is not stationary
9766 JP Equity and 9984 JP Equity pair is not stationary
9831 JP Equity and 9843 JP Equity pair is not stationary
9831 JP Equity and 9861 JP Equi

/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfSpread['spread'] = tpxData[pair[0]] - hedgeRatioParams[1] * tpxData[pair[1]] - hedgeRatioParams[0]
/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfSpread['spread'] = tpxData[pair[0]] - hedgeRatioParams[1] * tpxData[pair[1]] - hedgeRatioParams[0]
/var/folders/b1/x4vbkfvn2qlf297m4zgbkw5h0000gn/T/ipykernel_16588/2365185304.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a fut

In [45]:
valid = validPairs
# valid = pd.read_csv('validPairs2.csv', index_col=0, parse_dates=True)
valid

,spread_1332 JP Equity_1514 JP Equity,spread_1332 JP Equity_1802 JP Equity,spread_1332 JP Equity_1925 JP Equity,spread_1332 JP Equity_1959 JP Equity,spread_1332 JP Equity_3778 JP Equity,spread_1332 JP Equity_4578 JP Equity,spread_1514 JP Equity_1605 JP Equity,spread_1514 JP Equity_1662 JP Equity,spread_1514 JP Equity_1721 JP Equity,spread_1514 JP Equity_1801 JP Equity,...,spread_9602 JP Equity_9684 JP Equity,spread_9602 JP Equity_9719 JP Equity,spread_9603 JP Equity_9697 JP Equity,spread_9684 JP Equity_9843 JP Equity,spread_9684 JP Equity_9962 JP Equity,spread_9697 JP Equity_9983 JP Equity,spread_9719 JP Equity_9766 JP Equity,spread_9719 JP Equity_9861 JP Equity,spread_9735 JP Equity_9843 JP Equity,spread_9735 JP Equity_9861 JP Equity
Date,,,,,,,,,,,,,,,,,,,,,
1/1/2013,-308.074057,-34.790994,-1.166333,-68.340595,-269.518948,13.798589,-97.048839,-103.632830,89.376078,108.380669,...,-493.807673,-100.627553,-1732.572918,-211.251101,-672.039185,358.257603,-503.681068,-412.387571,-1704.203302,-2209.337779
2/1/2013,-308.074057,-34.790994,-1.166333,-68.340595,-269.518948,13.798589,-97.048839,-103.632830,89.376078,108.380669,...,-493.807673,-100.627553,-1732.572918,-211.251101,-672.039185,358.257603,-503.681068,-412.387571,-1704.203302,-2209.337779
3/1/2013,-308.074057,-34.790994,-1.166333,-68.340595,-269.518948,13.798589,-97.048839,-103.632830,89.376078,108.380669,...,-493.807673,-100.627553,-1732.572918,-211.251101,-672.039185,358.257603,-503.681068,-412.387571,-1704.203302,-2209.337779
4/1/2013,-303.533256,-35.114905,-11.786750,-64.173387,-264.975229,10.865337,-100.727097,-92.632573,89.134757,106.053942,...,-497.250505,-113.483180,-1774.573204,-220.504951,-697.413372,350.462463,-497.473525,-411.943293,-1690.481085,-2199.168069
7/1/2013,-309.288452,-36.931814,-9.996489,-66.132397,-268.225097,7.144132,-81.629274,-84.632687,106.479446,117.573194,...,-500.807673,-125.338808,-1722.073634,-214.002384,-674.171926,336.596029,-485.469770,-414.232449,-1671.295896,-2199.896534
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27/5/2024,106.744011,-116.780522,165.521991,55.969263,-116.708442,66.838794,540.178316,538.347169,1075.626706,1018.361880,...,883.998290,-1235.719874,9.388677,-575.865600,47.940812,-219.717113,-1.567861,471.895552,940.257014,-143.850763
28/5/2024,90.467222,-138.198755,157.238535,-2.802389,-119.976533,58.138794,558.414844,559.346827,1110.868027,1036.939457,...,979.328596,-1118.387168,-13.314384,-611.225844,-187.393415,-258.814683,-58.011195,462.085307,929.321849,-127.733252
29/5/2024,86.535194,-143.048156,152.849947,-8.342439,-119.462513,44.853853,467.626153,461.346314,1039.833312,959.823942,...,925.018244,-1196.283337,-19.671283,-666.930931,-279.981335,-167.967049,-177.039264,443.985906,904.562654,-190.344206


In [46]:
validPairs.to_csv("validPairs3.csv")

### Strategy

Enter when price goes above 1 std.

Cut loss when price goes above 1.5 std.

Take profit when spread goes to 0 or change sign.


In [47]:
rollingWindow = 262
cutLossSd = 2

In [48]:
pairsOutcome = {}

In [49]:
for pair in validPairsList:
    df = pd.DataFrame()

    #Calculate Standard Deviations
    df['spread'] = valid[f'spread_{pair[0]}_{pair[1]}']
    df['mid'] =  df['spread'].rolling(rollingWindow).mean()
    df['1sd high'] = df['spread'].rolling(rollingWindow).mean() + df['spread'].rolling(rollingWindow).std()
    df['1sd low'] = df['spread'].rolling(rollingWindow).mean() - df['spread'].rolling(rollingWindow).std()
    df['2sd high'] = df['spread'].rolling(rollingWindow).mean() + df['spread'].rolling(rollingWindow).std() * cutLossSd
    df['2sd low'] = df['spread'].rolling(rollingWindow).mean() - df['spread'].rolling(rollingWindow).std() * cutLossSd
    df['position'] = 0

    return_df = (tpxData / tpxData.shift(1)) - 1

    df.loc[(df['spread'] > df['1sd high']) & (df['spread'] < df['2sd high']), 'position'] = -1
    df.loc[(df['spread']< df['1sd low']) & (df['spread'] > df['2sd high']), 'position'] = 1

    #Calculate PnL
    df[f'{pair[0]} position'] = df['position']
    df[f'{pair[1]} position'] = df['position'] * -1
    df['dailypnl'] = df[f'{pair[1]} position']*return_df[f'{pair[1]}'].shift(-1) + df[f'{pair[0]} position']*return_df[f'{pair[0]}'].shift(-1)
    df['cumpnl'] = df['dailypnl'].cumsum()

    pairsOutcome[f'{pair[0]} {pair[1]}'] = df


In [58]:
pairsOutcome['1514 JP Equity 1605 JP Equity'].loc[pairsOutcome['1514 JP Equity 1605 JP Equity']['position'] != 0]

,spread,mid,1sd high,1sd low,2sd high,2sd low,position,1514 JP Equity position,1605 JP Equity position,dailypnl,cumpnl
Date,,,,,,,,,,,
28/11/2014,-116.610639,-202.619600,-132.220736,-273.018464,-61.821872,-343.417327,-1,-1,1,-0.014991,-0.014991
1/12/2014,-96.749343,-202.751863,-132.585855,-272.917872,-62.419847,-343.083880,-1,-1,1,0.040340,0.025349
2/12/2014,-120.178465,-202.992442,-133.220634,-272.764251,-63.448825,-342.536060,-1,-1,1,-0.001192,0.024157
3/12/2014,-119.326726,-203.238440,-133.878450,-272.598430,-64.518460,-341.958419,-1,-1,1,0.030275,0.054432
5/12/2014,-134.522372,-203.785038,-135.252865,-272.317210,-66.720693,-340.849382,-1,-1,1,-0.013261,0.041172
...,...,...,...,...,...,...,...,...,...,...,...
20/3/2024,1169.885245,241.953377,1001.951128,-518.044374,1761.948879,-1278.042125,-1,-1,1,0.037796,0.334627
21/3/2024,1108.184820,245.987683,1007.772269,-515.796904,1769.556856,-1277.581491,-1,-1,1,0.041110,0.375737
22/3/2024,1020.516119,249.755403,1012.924292,-513.413486,1776.093181,-1276.582375,-1,-1,1,0.093462,0.469199
